# 98 — Dashboard export

Write `dashboard_data.parquet` for the React dashboard. Layout: one row per land grid cell, with

- underscore-prefixed metadata columns (`_lat`, `_lon`, `_country_code`, `_continent`) that the dashboard treats as non-scoring context,
- every percentile-normalized layer from `normalized.nc` **except** `temperature_pleasantness`, `precipitation_balance`, and `population_density` — those three are replaced by profile-based variants,
- for each of the three replaced layers, one column per profile in `TEMP_PROFILES` / `PRECIP_PROFILES` / `DENSITY_PROFILES`, suffixed `_p1`, `_p2`, …, and percentile-normalized so they share the same `[0, 1]` uniform distribution as the other layers.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
from scipy.stats import rankdata

from common import (
    PROCESSED_DIR,
    compute_population_density_score,
    compute_precipitation_balance,
    compute_temperature_pleasantness,
    load_raw_scoring_inputs,
)

## Profiles

`(ideal, tolerance)` for temperature and precipitation; `((min, max), tolerance_decades)` for density where `None` means unbounded on that side — wilderness is a pure upper bound (empty cells still score 1), urban a pure lower bound.

In [ ]:
TEMP_PROFILES = {
    'Icy (5°C ±8)':          (0.0, 8.0),
    'Cool (15°C ±8)':         (15.0, 8.0),
    'Temperate (20°C ±10)':   (20.0, 10.0),
    'Warm (25°C ±8)':         (25.0, 8.0),
    'Tropical (27°C ±5)':     (27.0, 5.0),
}

PRECIP_PROFILES = {
    'Arid (20mm ±20)':       (20.0, 20.0),
    'Balanced (80mm ±60)':   (80.0, 60.0),
    'Wet (150mm ±60)':       (150.0, 60.0),
    'Very wet (250mm ±80)':  (250.0, 80.0),
}

DENSITY_PROFILES = {
    'Wilderness (≤1/km² ±1dec)':     ((None, 1),    1.0),
    'Rural (10–300/km² ±1dec)':       ((10, 300),    1.0),
    'Suburban (200–1500/km² ±1dec)':  ((200, 1500),  1.0),
    'Urban (≥1500/km² ±1dec)':         ((1500, None), 1.0),
}

## Helpers

`percentile_rank` mirrors the transform in `92_normalization.ipynb` so the profile columns share the same uniform `[0, 1]` distribution as the base layers already in `normalized.nc`.

In [ ]:
def percentile_rank(da):
    """Rank-transform a DataArray to percentiles in ``[0, 1]``.

    Ties are averaged. NaN cells are preserved and excluded from the ranking.
    Shares dims, coords and name with the input so it can be dropped into any
    consumer that previously took the raw layer.
    """
    values = da.values.astype(float)
    flat = values.ravel()
    mask = np.isfinite(flat)
    out = np.full_like(flat, np.nan)
    n = int(mask.sum())
    if n > 1:
        out[mask] = (rankdata(flat[mask], method='average') - 1) / (n - 1)
    elif n == 1:
        out[mask] = 0.0
    return da.copy(data=out.reshape(values.shape))

## Base layers

Start from `normalized.nc` (percentile-transformed by `92_normalization.ipynb`) and drop the three layers we're replacing with profile-based variants.

In [ ]:
REPLACED = ('temperature_pleasantness', 'precipitation_balance', 'population_density')

ds = xr.open_dataset(PROCESSED_DIR / 'normalized.nc')
base = ds.drop_vars([v for v in REPLACED if v in ds.data_vars])
list(base.data_vars)

## Profile layers

For each profile, compute the raw comfort score with the helper in `common.py`, then percentile-rank so the result is comparable to the other columns. Column names are `<base>_p1`, `_p2`, … in dict-iteration order — the human-readable labels stay in this notebook (and are mirrored in the dashboard).

In [ ]:
temp_da, precip_da, density_da = load_raw_scoring_inputs()

profile_layers = {}

for i, (label, (ideal, tol)) in enumerate(TEMP_PROFILES.items(), start=1):
    score = compute_temperature_pleasantness(temp_da, ideal_temp=ideal, tolerance=tol)
    profile_layers[f'temperature_pleasantness_p{i}'] = percentile_rank(score)
    print(f'temperature_pleasantness_p{i}  <- {label}')

for i, (label, (ideal, tol)) in enumerate(PRECIP_PROFILES.items(), start=1):
    score = compute_precipitation_balance(precip_da, ideal_monthly_mm=ideal, tolerance_mm=tol)
    profile_layers[f'precipitation_balance_p{i}'] = percentile_rank(score)
    print(f'precipitation_balance_p{i}     <- {label}')

for i, (label, (rng, tol_dec)) in enumerate(DENSITY_PROFILES.items(), start=1):
    score = compute_population_density_score(density_da, density_range=rng, tolerance_decades=tol_dec)
    profile_layers[f'population_density_p{i}'] = percentile_rank(score)
    print(f'population_density_p{i}        <- {label}')

## Merge and flatten

Combine the base and profile layers on the shared `(lat, lon)` grid, flatten to one row per cell, and drop cells that are NaN across every value column (pure ocean / coverage gaps) before the spatial join.

In [ ]:
combined = xr.Dataset({**base.data_vars, **profile_layers})
df = combined.to_dataframe().reset_index()

value_cols = [c for c in df.columns if c not in ('lat', 'lon')]
df = df.dropna(subset=value_cols, how='all').reset_index(drop=True)
len(df)

## Country & continent

Point-in-polygon join against Natural Earth 50m admin_0 boundaries — same pattern as `94_post_processing.ipynb`. Ocean / disputed cells get NaN for both columns.

In [ ]:
import geopandas as gpd
import httpx

from common import RAW_DIR

# Reuse the cached Natural Earth shapefile downloaded by 94_post_processing;
# fetch it here too so this notebook can run standalone.
NE_URL = 'https://naciscdn.org/naturalearth/50m/cultural/ne_50m_admin_0_countries.zip'
ne_zip = RAW_DIR / 'post_processing' / 'ne_50m_admin_0_countries.zip'

if not ne_zip.exists():
    print(f'downloading {NE_URL}')
    ne_zip.parent.mkdir(parents=True, exist_ok=True)
    r = httpx.get(NE_URL, headers={'User-Agent': 'Mozilla/5.0'}, timeout=180, follow_redirects=True)
    r.raise_for_status()
    ne_zip.write_bytes(r.content)

countries = (
    gpd.read_file(f'zip://{ne_zip}')
    .to_crs('EPSG:4326')
    [['ISO_A3_EH', 'CONTINENT', 'geometry']]
    .rename(columns={'ISO_A3_EH': 'country_code', 'CONTINENT': 'continent'})
)

points = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df['lon'], df['lat']),
    crs='EPSG:4326',
)
joined = points.sjoin(countries, how='left', predicate='within')
# Points on shared borders can duplicate; keep the first hit so row count is preserved.
joined = joined[~joined.index.duplicated(keep='first')]
df = pd.DataFrame(joined.drop(columns=['geometry', 'index_right']))

n_missing = df['country_code'].isna().sum()
print(f'{len(df):,} cells; {n_missing:,} ({n_missing / len(df):.1%}) with no country match')

## Save

Rename metadata columns to the underscore-prefixed form the dashboard expects (`_lat`, `_lon`, `_country_code`, `_continent`) and write.

In [ ]:
df = df.rename(columns={
    'lat': '_lat',
    'lon': '_lon',
    'country_code': '_country_code',
    'continent': '_continent',
})

out = PROCESSED_DIR / 'dashboard_data.parquet'
df.to_parquet(out, index=False)
print(f'wrote {out} ({out.stat().st_size / 1024**2:.1f} MB, {len(df):,} rows, {len(df.columns)} cols)')
df.head()